<a id="unit-u.897aa48b985e43e5991663a8bb17949c"></a>

<div>
  <div>01-ЗЕРТХАНАЛЫҚ ЖҰМЫС · APACHE DORIS КӨМЕГІМЕН НАҚТЫ УАҚЫТТАҒЫ АНАЛИТИКА</div>
  <div>Doris іске қосу және бастапқы деректер жиынын құру</div>
  <p>Сақтау мен есептеуі біріктірілген, қайта пайдаланылатын бір түйінді Doris сынақ ортасын іске қосып, S3 ішіндегі интернет-сауда оқиғаларының жиынын зерттеңіз және тексерілген 10,158,080 жолдық ішкі кесте құрыңыз.</p>
  <span>Doris 4.1.3 · FE · BE · S3 TVF · Ішкі кесте · Талдамалық сұрау</span>
</div>

**1-модуль: Apache Doris жүйесіне кіріспе** · [Сабақ](course.md) · [Тест](quiz01_doris_fundamentals.ipynb)

Шамамен 30 минут жоспарлаңыз; алғашқы орындауда көлемі шамамен 1.9 GB Doris бейнесі де жүктеледі.

Осы зертханалық жұмыстың соңында жұмыс істеп тұрған Doris ортасы, 10 миллионнан астам жолы бар `events` кестесі және осы кестеден алынған талдамалық нәтижелер болады. Ұяшықтарды ретімен орындаңыз.

<a id="unit-u.e51c1b564ccf446a85913d6720194862"></a>

<a id="initialize-the-lab"></a>

### Зертханалық жұмысты бастапқы баптау

1-бөлімді бастамас бұрын келесі баптау ұяшығын орындаңыз. Ол жергілікті көмекші құралды жүктейді, нәтиже стильдерін орнатады және кейінгі барлық код ұяшығы пайдаланатын `lab` нысанын жасайды.

Jupyter ядросын қайта іске қосқан сайын оны қайта орындау керек. Ол Docker іске қоспайды, S3 қызметіне қатынамайды және деректерді **өзгертпейді**. Кейін SQL ұяшығы орындалғанда, көмекші құрал `127.0.0.1:19030` мекенжайындағы бар Doris сынақ ортасына автоматты түрде қайта қосылып, `doris_course` дерекқор контекстін қалпына келтіре алады. Алғашқы орындауда ортаны құрып, іске қосу үшін 2-бөлімді аяқтаңыз. Баптау сәтті аяқталғанда жасыл **Lab tools are ready** (зертханалық құралдар дайын) хабарламасы шығады. Оны көрмейінше жалғастырмаңыз.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT);


<a id="unit-u.82f737c1543f418a8aee9c56937ed2cc"></a>

<a id="1-understand-the-sandbox-and-dataset"></a>

## 1. Сынақ ортасы мен деректер жиынын түсіну

Ноутбук Frontend (FE) компонентіне оның MySQL-мен үйлесімді сұрау порты арқылы қосылады. FE SQL қабылдап, талдайды, үлестірілген сұрау жоспарын құрып, оңтайландырады және жоспар фрагменттерін Backend (BE) түйіндеріне тағайындайды. BE түйіндері фрагменттерді орындайды және сақтау мен есептеуі біріктірілген осы архитектурада ішкі кесте деректерін сақтайды.

| Компонент | Осы жұмыстағы міндеті |
|---|---|
| Ноутбук клиенті | SQL сұрауын хосттың 19030 портына жібереді, Docker оны FE сұрау порты 9030-ға бағыттайды. |
| Frontend (FE) | SQL қабылдап, талдайды, үлестірілген сұрау жоспарын құрып, оңтайландырады және оның фрагменттерін тағайындайды. |
| Backend (BE) | Тағайындалған жоспар фрагменттерін орындайды және `events` ішкі кестесінің деректерін сақтайды. |

Ресми All-in-One бейнесі бір контейнерде бір FE және бір BE іске қосады. Бұл **сақтау мен есептеуі біріктірілген бір түйінді Doris сынақ ортасы**, өндірістік орналастыру емес. Docker атаулы томдары контейнер тоқтағаннан кейін дерекқор метадеректері мен кесте деректерін сақтайды.

Деректер жиынында интернет-сауда әрекеттерінің 10,158,080 жазбасы бар. Әр жол `view`, `cart` немесе `purchase` оқиғасын сипаттайды. `region` мәндері жасанды және әр пайдаланушы үшін тұрақты; олар нақты орналасқан жерді білдірмейді. `revenue` сатып алулар үшін өнім бағасын, ал басқа оқиға түрлері үшін `0.00` сақтайды.

<div><strong>Бастамас бұрын</strong>Docker үшін кемінде 4 CPU ядросы мен 8 GB жады бөліп, дискіде кемінде 20 GB бос орын қалдырыңыз. Зертханалық жұмыс Docker Hub сайтынан Doris бейнесін (шамамен 1.9 GB) және AWS S3 қызметінің <code>us-east-1</code> аймағынан 72 MiB дерек жүктейді, сондықтан алғашқы орындау бірнеше минут алуы мүмкін. Docker Hub немесе AWS баяу не бұғатталған желіде алдымен курс README файлындағы Орта бөлімін оқыңыз.</div>

<a id="unit-u.73d68c559d3d4efe8c84b3b20f6c84db"></a>

<a id="dataset-source-and-attribution"></a>

### Деректер жиынының көзі және авторлық деректер

Бұл зертханалық жұмыс Michael Kechinov жариялаған және **REES46 Marketing Platform** ұсынған **eCommerce behavior data from multi category store** жиынының түрлендірілген ішкі жиынын пайдаланады. Интернет-сауда әрекеттерінің бастапқы деректерін ашық ұсынған REES46 платформасына алғыс білдіреміз.

- Бастапқы жиын: [eCommerce behavior data from multi category store](https://www.kaggle.com/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store/data)
- Жиын ұсынушысы: [REES46 ашық деректер жиындары](https://rees46.com/en/datasets)
- Курс файлдарын дайындауға пайдаланылған ашық Parquet айнасы: [Firebolt E-Commerce Analytics Primer](https://www.firebolt.io/free-sample-datasets/e-commerce)

Осы жұмыста оқылатын файлдар бастапқы жарияланымның өзі емес, түрлендірілген көшірмесі. Түрлендіру `event_time`, `event_type`, `product_id` және `user_id` өрістерін сақтайды; тұрақты `event_id` жасайды; әр пайдаланушыға сегіз жасанды `region` мәнінің бірін тағайындайды; бастапқы бағаны тек `purchase` оқиғалары үшін `revenue` ретінде сақтайды. Жасанды аймақтар пайдаланушылардың нақты орналасқан жерін білдірмейді.

| Курс көшірмесінің орны | |
|---|---|
| Bucket және аймақ | `doris-public-dataset`, AWS `us-east-1` |
| Нысандар | `doris-course/events/v1/events-00000.parquet` … `events-00039.parquet` |
| Көлемі | Жалпы шамамен 72 MiB болатын 40 Parquet файлы, 10,158,080 жол |
| Қолжеткізу | Ашық және тек оқуға арналған. AWS тіркелгісі не кіру деректері қажет емес. |

<a id="unit-u.68670dde462748e1a4501262989ce1f2"></a>

<a id="2-prepare-the-environment"></a>

## 2. Ортаны дайындау

Бұл бөлім Docker тексеріп, қайта пайдаланылатын сақтау орнын құрады, Doris 4.1.3 іске қосады және FE компонентіне қосылады. Командалар macOS Apple Silicon және Linux x86_64 жүйелерін қолдайды; Windows жүйесінде Jupyter ортасын WSL 2 дистрибутивінің ішінде іске қосыңыз, сонда командалар Linux x86_64 жүйесін көреді. Docker бейне архитектурасын автоматты түрде таңдайды, сондықтан Apple Silicon жүйесінде `--platform linux/amd64` қоспаңыз.

<div>
<table>
<tr>
<td><strong>1 · Docker тексеру</strong><br><span>Клиент пен жұмыс істеп тұрған демонды тексеру</span></td>
<td>&#8594;</td>
<td><strong>2 · Порттарды тексеру</strong><br><span>Хосттың 19030, 18030 және 18040 порттары бос екенін тексеру</span></td>
<td>&#8594;</td>
<td><strong>3 · Сақтау орнын құру</strong><br><span>Желі мен атаулы томдарды құру</span></td>
<td>&#8594;</td>
<td><strong>4 · Бейнені алу</strong><br><span>Бекітілген Doris 4.1.3 бейнесін қайта пайдалану немесе жүктеп алу</span></td>
<td>&#8594;</td>
<td><strong>5 · Doris іске қосу</strong><br><span>Контейнерді іске қосып, сау күйін күту</span></td>
<td>&#8594;</td>
<td><strong>6 · Түйіндерді тексеру</strong><br><span>FE компонентіне қосылып, FE/BE күйін сұрау</span></td>
</tr>
</table>
</div>

Келесі ұяшықтың басындағы айнымалылар бейне, контейнер, желі және том атауларын береді. Хост порттары `DORIS_RTA_QUERY_PORT`, `DORIS_RTA_HTTP_PORT` және `DORIS_RTA_BE_HTTP_PORT` арқылы алынады (әдепкіде 19030, 18030 және 18040); кейінгі қосылым ұяшықтары да сол айнымалыларды оқиды. Басқа порт керек болса, оларды Jupyter іске қосылмай тұрып орнатыңыз. BE веб-сервері контейнер ішінде де хост портын тыңдайды, сондықтан FE Stream Load сұрауын BE компонентіне қайта бағыттағанда қайтаратын мекенжай компьютеріңізден де жұмыс істейді.

**Docker орнату және сақтау орны**

| Хост | Осы жұмыстағы Docker ортасы | Автоматты таңдалатын Doris бейнесі |
|---|---|---|
| macOS Apple Silicon | Apple silicon үшін Docker Desktop for Mac | `linux/arm64` |
| Linux x86_64 | Linux дистрибутивіне арналған Docker Engine | `linux/amd64` |
| WSL 2 бар Windows | WSL 2 интеграциясы бар Docker Desktop; Jupyter ортасын WSL дистрибутивінде іске қосыңыз | `linux/amd64` |

Барлық хост бірдей `apache/doris:all-in-one-4.1.3` тегін жүктейді. Тегте екі архитектура да бар, Docker хостқа сәйкесін таңдайды. Intel Mac немесе arm64 Linux сияқты басқа операциялық жүйе мен архитектура үйлесімдері Docker ресурстары жасалмай тұрып тоқтайды: курс оларда орындалмаған. Курстың README файлындағы Орта бөлімі сыналған жүйелерді көрсетеді.

Бастамас бұрын Docker орнатыңыз. Орнату әкімші рұқсатын, Docker Desktop алғашқы келісімін не ұйымға тән баптауды қажет етуі мүмкін. macOS жүйесінде орта ұяшығы демоны тоқтаған орнатулы Docker Desktop қолданбасын аша алады.

Docker бейнелері ноутбук каталогына жүктелмейді. Docker оларды өзінің басқарылатын деректер аймағында сақтайды; атаулы томдар FE метадеректері мен BE кесте деректерін сақтайды. Том атаулары: `doris-rta-fe-meta` және `doris-rta-be-storage`. Бұл жұмыс үшін олардың физикалық орнын әдетте өзгертпеу керек. Docker басқаратын дискіде бос орын жетпесе, жұмысты бастамас бұрын macOS жүйесінде Docker Desktop диск бейнесінің орнын немесе Linux жүйесінде Docker Engine data-root орнын өзгертіңіз.

In [ ]:
lab.shell(r"""
set -euo pipefail

DORIS_IMAGE="apache/doris:all-in-one-4.1.3"
DORIS_CONTAINER="doris-rta"
DORIS_NETWORK="doris-rta"
FE_VOLUME="doris-rta-fe-meta"
BE_VOLUME="doris-rta-be-storage"
# Host ports. To use other ports, set these variables before starting Jupyter.
QUERY_PORT="${DORIS_RTA_QUERY_PORT:-19030}"      # forwards to FE query port 9030
HTTP_PORT="${DORIS_RTA_HTTP_PORT:-18030}"        # forwards to FE HTTP port 8030
BE_HTTP_PORT="${DORIS_RTA_BE_HTTP_PORT:-18040}"  # BE web server, same port inside the container

echo "[1/6] Verify the host, Docker client, context, and daemon"
HOST_OS=$(uname -s)
HOST_ARCH=$(uname -m)
case "${HOST_OS}/${HOST_ARCH}" in
  Darwin/arm64|Darwin/aarch64)
    ;;
  Linux/x86_64|Linux/amd64)
    ;;
  *)
    echo "Unsupported host: ${HOST_OS}/${HOST_ARCH}"
    echo "This course runs on macOS Apple Silicon and Linux x86_64 (on Windows, inside WSL 2)."
    echo "It has not been run on this combination; see Environment in the course README."
    exit 2
    ;;
esac

if ! command -v docker >/dev/null 2>&1; then
  echo "Docker CLI was not found. Install Docker for ${HOST_OS}/${HOST_ARCH}, restart the terminal or kernel, and rerun this cell."
  exit 3
fi

echo "docker_cli=$(docker --version)"
echo "docker_context=$(docker context show 2>/dev/null || echo unknown)"
if [ -n "${DOCKER_HOST:-}" ]; then
  echo "DOCKER_HOST=${DOCKER_HOST}"
fi

if ! docker info >/dev/null 2>&1; then
  if [ "$HOST_OS" = "Darwin" ]; then
    if [ ! -d /Applications/Docker.app ]; then
      echo "Docker Desktop is not installed in /Applications/Docker.app."
      echo "Install the Apple silicon build of Docker Desktop, then rerun this cell."
      exit 4
    fi
    echo "Docker Desktop is installed but its daemon is not ready. Opening Docker Desktop..."
    open -a Docker
    for attempt in $(seq 1 60); do
      if docker info >/dev/null 2>&1; then
        break
      fi
      if [ $((attempt % 5)) -eq 0 ]; then
        echo "Waiting for Docker Desktop (${attempt}/60)..."
      fi
      sleep 2
    done
  else
    echo "Docker Engine is installed but its daemon is unavailable."
    echo "Start it in a terminal, commonly with: sudo systemctl start docker"
    echo "For rootless Docker, commonly use: systemctl --user start docker"
    exit 5
  fi
fi

if ! docker info >/dev/null 2>&1; then
  echo "Docker did not become ready. Complete any first-run dialog, check the selected context and DOCKER_HOST, then rerun this cell."
  exit 6
fi

docker version --format 'client={{.Client.Version}} server={{.Server.Version}}'
docker info --format 'cpus={{.NCPU}} memory={{.MemTotal}} docker_root={{.DockerRootDir}} os={{.OperatingSystem}} arch={{.Architecture}}'

echo "[2/6] Check that the host ports are free"
echo "query=${QUERY_PORT} fe_http=${HTTP_PORT} be_http=${BE_HTTP_PORT}"
if [ "$(docker inspect --format '{{.State.Running}}' "$DORIS_CONTAINER" 2>/dev/null || true)" != "true" ]; then
  for PORT in "$QUERY_PORT" "$HTTP_PORT" "$BE_HTTP_PORT"; do
    if (exec 3<>"/dev/tcp/127.0.0.1/${PORT}") 2>/dev/null; then
      echo "Host port ${PORT} is already in use by another program."
      if command -v lsof >/dev/null 2>&1; then
        lsof -nP -iTCP:"${PORT}" -sTCP:LISTEN || true
      fi
      echo "Stop that program, or choose free ports before starting Jupyter, for example:"
      echo "  export DORIS_RTA_QUERY_PORT=29030 DORIS_RTA_HTTP_PORT=28030 DORIS_RTA_BE_HTTP_PORT=28040"
      exit 9
    fi
  done
fi

echo "[3/6] Create or reuse the Docker network and named volumes"
docker network inspect "$DORIS_NETWORK" >/dev/null 2>&1 || docker network create "$DORIS_NETWORK"
docker volume inspect "$FE_VOLUME" >/dev/null 2>&1 || docker volume create "$FE_VOLUME"
docker volume inspect "$BE_VOLUME" >/dev/null 2>&1 || docker volume create "$BE_VOLUME"

echo "[4/6] Reuse or pull the pinned Doris image"
docker image inspect "$DORIS_IMAGE" >/dev/null 2>&1 || docker pull "$DORIS_IMAGE"

echo "[5/6] Start or reuse the Doris container"
if docker container inspect "$DORIS_CONTAINER" >/dev/null 2>&1; then
  CURRENT_IMAGE=$(docker inspect --format '{{.Config.Image}}' "$DORIS_CONTAINER")
  if [ "$CURRENT_IMAGE" != "$DORIS_IMAGE" ]; then
    echo "Container $DORIS_CONTAINER uses $CURRENT_IMAGE, expected $DORIS_IMAGE"
    echo "The existing container was left unchanged. Rename it or choose another course container name."
    exit 7
  fi
  CURRENT_NETWORKS=$(docker inspect --format '{{range $name, $_ := .NetworkSettings.Networks}}{{println $name}}{{end}}' "$DORIS_CONTAINER")
  CURRENT_MOUNTS=$(docker inspect --format '{{range .Mounts}}{{printf "%s|%s\n" .Name .Destination}}{{end}}' "$DORIS_CONTAINER")
  # Read the configured bindings; `docker port` shows nothing while the container is stopped.
  CURRENT_PORTS=$(docker inspect --format '{{range $port, $bindings := .HostConfig.PortBindings}}{{range $bindings}}{{printf "%s %s:%s\n" $port .HostIp .HostPort}}{{end}}{{end}}' "$DORIS_CONTAINER")
  if ! grep -Fxq "$DORIS_NETWORK" <<<"$CURRENT_NETWORKS" \
     || ! grep -Fxq "$FE_VOLUME|/opt/apache-doris/fe/doris-meta" <<<"$CURRENT_MOUNTS" \
     || ! grep -Fxq "$BE_VOLUME|/opt/apache-doris/be/storage" <<<"$CURRENT_MOUNTS" \
     || ! grep -Fxq "9030/tcp 127.0.0.1:${QUERY_PORT}" <<<"$CURRENT_PORTS" \
     || ! grep -Fxq "8030/tcp 127.0.0.1:${HTTP_PORT}" <<<"$CURRENT_PORTS" \
     || ! grep -Fxq "${BE_HTTP_PORT}/tcp 127.0.0.1:${BE_HTTP_PORT}" <<<"$CURRENT_PORTS"; then
    echo "Container $DORIS_CONTAINER does not use the expected network, named volumes, or localhost port bindings."
    echo "The existing container was left unchanged. If you changed the DORIS_RTA_*_PORT variables,"
    echo "remove it with: docker rm -f $DORIS_CONTAINER (the named volumes keep the data), then rerun this cell."
    exit 8
  fi
  if [ "$(docker inspect --format '{{.State.Status}}' "$DORIS_CONTAINER")" != "running" ]; then
    docker start "$DORIS_CONTAINER"
  fi
else
  docker run -d \
    --name "$DORIS_CONTAINER" \
    --network "$DORIS_NETWORK" \
    -e "BE_HTTP_PORT=${BE_HTTP_PORT}" \
    -e "BE_CONFIG_EXTRA=webserver_port = ${BE_HTTP_PORT}" \
    -p "127.0.0.1:${QUERY_PORT}:9030" \
    -p "127.0.0.1:${HTTP_PORT}:8030" \
    -p "127.0.0.1:${BE_HTTP_PORT}:${BE_HTTP_PORT}" \
    -v "$FE_VOLUME:/opt/apache-doris/fe/doris-meta" \
    -v "$BE_VOLUME:/opt/apache-doris/be/storage" \
    "$DORIS_IMAGE"
fi

echo "[6/6] Show the running course container"
docker ps --filter "name=^/${DORIS_CONTAINER}$"
""", title="Prepare the Doris environment");

<a id="unit-u.bf8eb214de8a4feeb0affdf1877d2139"></a>

**Doris жүйесіне қосылу және түйіндерді тексеру**

Жоғарыдағы Bash командалары контейнерді іске қосады, бірақ жұмыс істеп тұрған контейнер сұрау қабылдауға әлі дайын болмауы мүмкін. Келесі ұяшық Docker `healthy` күйін көрсеткенше күтіп, хосттың 19030 порты арқылы FE компонентіне MySQL-мен үйлесімді протокол қосылымын ашады да, FE және BE күй кестелеріне сұрау жасайды. Осылайша дерекқор нысандары жасалмай тұрып екі Doris қызметінің де дайын екені расталады.

In [ ]:
lab.connect()

lab.sql("SELECT CURRENT_USER() AS current_user", title="Connection")
lab.sql("SHOW FRONTENDS", title="Frontend status")
lab.sql("SHOW BACKENDS", title="Backend status");


<a id="unit-u.9035b89e26b240b28e25f581cf3f7f47"></a>

**Күтілетін нәтиже**

- Docker командасы сәтті аяқталып, `doris-rta` контейнері `healthy` күйіне өтеді.
- `SHOW FRONTENDS` бір жол қайтарады: `Alive=true`, `Join=true` және `IsMaster=true`. Оның `Version` бағаны `doris-4.1.3-rc02-7126cf65d96` көрсетеді: 4.1.3 релизі осы құрастырылымға сәйкес. (`SELECT VERSION()` `5.7.99` қайтарады; бұл Doris нұсқасы емес, Doris клиенттерге хабарлайтын MySQL протоколының нұсқасы.)
- `SHOW BACKENDS` бір жол қайтарады: `Alive=true` және `SystemDecommissioned=false`.

**Жиі кездесетін мәселелер**

- macOS жүйесінде `server=` бос болса немесе белсенді Docker сокеті жоқ болса, ұяшық Docker Desktop ашып, екі минутқа дейін күтеді. Алғашқы диалогтарды аяқтап, қозғалтқыш дайын болғанша күтіңіз; күту уақыты өтсе, ұяшықты қайталаңыз.
- Linux `/var/run/docker.sock` сокетіне қосыла алмаса, Docker Engine қызметін `sudo systemctl start docker` арқылы іске қосыңыз. `permission denied` хабарламасы пайдаланушыңызға ұйым талап ететін Docker құқықтары керегін білдіреді.
- Хост портын басқа бағдарлама иеленсе және ол курс контейнеріне тиесілі болмаса, ұяшық тоқтап, сол бағдарламаны көрсетеді. Оны тоқтатыңыз немесе Jupyter іске қосылмай тұрып `DORIS_RTA_QUERY_PORT`, `DORIS_RTA_HTTP_PORT` және `DORIS_RTA_BE_HTTP_PORT` айнымалыларын бос порттарға орнатыңыз. Бұл жұмыс қатысы жоқ жергілікті Doris орнатылымын автоматты түрде қайта пайдаланбайды, себебі оның нұсқасы, топологиясы, сақтау орны және қайта іске қосылу тәртібі өзгеше болуы мүмкін.
- Бейне жүктелмесе, Docker Hub қолжетімділігін және прокси не VPN баптауларын тексеріңіз.
- Doris сау емес күйге өтсе, `docker logs --tail 200 doris-rta` тексеріңіз.
- `docker context show` немесе `DOCKER_HOST` қолжетімсіз қашықтағы қозғалтқышты көрсетсе, ұяшықты қайталамас бұрын дұрыс Docker контекстін таңдаңыз немесе `DOCKER_HOST` түзетіңіз.
- Docker жады не диск жетпейтінін хабарласа, бейнені жүктемес немесе деректерді импорттамас бұрын Docker ресурстарын не басқарылатын диск сыйымдылығын арттырыңыз.

<a id="unit-u.96ff9fc1c632400186fa0e7f2844a31b"></a>

<a id="3-create-the-course-database"></a>

## 3. Курс дерекқорын құру

Doris Catalog ішінде дерекқорлар, ал әр дерекқорда кестелер болады. Кірістірілген `internal` Catalog ішінде `doris_course` құрып, оны ағымдағы дерекқор ретінде таңдаңыз және қолжетімді екенін растаңыз.

In [ ]:
lab.execute("CREATE DATABASE IF NOT EXISTS doris_course")
lab.execute("USE doris_course")
lab.sql("SHOW DATABASES", title="Databases")
lab.sql("SELECT DATABASE() AS current_database", title="Current database");


<a id="unit-u.9328b3b57361479ea6780494758a4c24"></a>

**Күтілетін нәтиже:** `SHOW DATABASES` ішінде `doris_course` бар, ал `SELECT DATABASE()` `doris_course` қайтарады.

<a id="unit-u.050b3eb4685e49bc88eb576855863421"></a>

<a id="4-inspect-the-remote-dataset-in-s3"></a>

## 4. S3 ішіндегі қашықтағы деректер жиынын зерттеу

Amazon S3 Parquet файлдарын Doris жүйесінен тыс сақтайды. S3 кестелік функциясы (S3 TVF) алдымен Doris кестесін құрмай-ақ, сұрау арқылы сол файлдарды жолдар мен бағандар ретінде оқуға мүмкіндік береді.

Төмендегі `S3(...)` шақыруы толық SQL болып табылады; оны Doris жүйесіне қосылған кез келген MySQL клиентінде өзгертпей орындауға болады. Қасиеттер файлдарды және оларға қалай жетуді көрсетеді:

| Қасиет | Осы жұмыстағы мәні |
|---|---|
| `uri` | `*` үлгі таңбасымен сәйкес келетін 40 курс файлы |
| `s3.endpoint`, `s3.region` | Курс bucket үшін AWS соңғы нүктесі мен аймағы |
| `s3.credentials_provider_type` | `ANONYMOUS`: bucket ашық, сондықтан Doris қолтаңбасыз сұраулар жібереді және кіру деректерін іздемейді |
| `s3.connection.timeout`, `s3.connection.request.timeout` | `10000` және `30000` миллисекунд: Doris қосылым ашуға 10 секундқа дейін жұмсай алады, ал оқу кезінде 30 секунд бойы дерек келмеуі мүмкін. Әдепкі 1 және 3 секунд баяу желіде немесе прокси арқылы қате туғызуы мүмкін |
| `format` | `parquet` |

Жеке bucket үшін оның орнына `s3.access_key` және `s3.secret_key` қажет болар еді (жұмыс соңындағы S3 TVF анықтамасын қараңыз).

Үлгіде мыналар бар:

| Баған | Мағынасы |
|---|---|
| `event_time` | Оқиға болған уақыт |
| `event_id` | Оқиға жолының бірегей идентификаторы |
| `user_id` | Пайдаланушы идентификаторы |
| `event_type` | `view`, `cart` немесе `purchase` |
| `region` | Пайдаланушыға тұрақты тағайындалған жасанды аймақ |
| `product_id` | Өнім идентификаторы |
| `revenue` | Сатып алу бағасы; сатып алу емес оқиғалар үшін `0.00` |

In [ ]:
lab.sql("""
SELECT
    event_time,
    event_id,
    user_id,
    event_type,
    region,
    product_id,
    revenue
FROM S3(
    "uri" = "s3://doris-public-dataset/doris-course/events/v1/events-*.parquet",
    "s3.endpoint" = "https://s3.us-east-1.amazonaws.com",
    "s3.region" = "us-east-1",
    "s3.credentials_provider_type" = "ANONYMOUS",
    "s3.connection.timeout" = "10000",
    "s3.connection.request.timeout" = "30000",
    "format" = "parquet"
)
LIMIT 10
""", title="Ten event rows read from S3");

<a id="unit-u.e085f484c47649d5a6b255d37d9909b9"></a>

**Күтілетін нәтиже:** он оқиға жолы бар бір кесте. Бұл жолдар S3 ішіндегі курс көшірмесінен тікелей оқылады; Doris ішкі кестесіне әлі ешбір жол жазылған жоқ.

<a id="unit-u.3a4ca3aab38c4ccda80bdf3caccc3a0a"></a>

<a id="5-create-the-first-doris-internal-table"></a>

## 5. Алғашқы Doris ішкі кестесін құру

Ішкі кесте деректерін BE сақтау орнында сақтайды. Бір BE бар осы сынақ ортасы үшін баған атауларын, дерек типтерін, NULL мәніне рұқсатты және бір репликаны анықтаңыз. Кейін Doris қайтарған кесте анықтамасын зерттеңіз.

In [ ]:
lab.execute("""
CREATE TABLE IF NOT EXISTS events (
    event_time DATETIME NOT NULL,
    event_id BIGINT NOT NULL,
    user_id BIGINT NOT NULL,
    event_type VARCHAR(32) NOT NULL,
    region VARCHAR(16) NOT NULL,
    product_id BIGINT NOT NULL,
    revenue DECIMAL(12, 2) NOT NULL DEFAULT "0.00"
)
PROPERTIES (
    "replication_num" = "1"
)
""")

lab.sql(
    "DESC events",
    title="events column schema",
    columns=["Field", "Type", "Null", "Key", "Default"],
)

lab.sql(
    "SHOW CREATE TABLE events",
    title="Resolved table design",
    metadata_view="design",
);

<a id="unit-u.1c7b8a415af045a5af29ff3d102c2b1e"></a>

**Күтілетін нәтиже**

Баған схемасында жоғарыда жасалған әр өріс үшін бір жолдан, барлығы жеті жол бар. Құрылымның жиынтық көрінісі осы қысқа DDL негізінде Doris анықтаған физикалық таңдауларды ғана көрсетеді:

| Құрылым таңдауы | Күтілетін мән |
|---|---|
| Кесте моделі | Duplicate Key моделі |
| Сұрыптау кілті | `event_time, event_id, user_id` |
| Бөлімдеу | Айқын бөлімдеу жоқ |
| Үлестіру | RANDOM |
| Бір бөлімдегі bucket саны | AUTO |
| Репликаларды бөлу | Әр tablet үшін 1 реплика · tag.location.default |

Алдыңғы орындаудан осы тұрақты кестеде жолдар қалса, келесі бөлім жаңа импорт алдында оларды жояды.

<a id="unit-u.5e7ff9d586ec4df69a25c7053bc4d419"></a>

<a id="6-reload-and-validate-the-baseline-dataset"></a>

## 6. Бастапқы деректер жиынын қайта жүктеу және тексеру

`TRUNCATE TABLE` алдыңғы орындаудан қалған жолдарды жойып, `events` кестесінің анықтамасын сақтайды. Кейінгі `INSERT INTO ... SELECT` S3 TVF арқылы сәйкес келетін барлық Parquet файлын оқып, толық деректер жиынын Doris жүйесіне қайта жазады. Екі операторды да орындау әр жұмыста нақты жүктеу шығынын көрсетеді. Doris шамамен 72 MiB жүктейді; желіге қарай енгізу әдетте он секундтан екі минутқа дейін созылады. Баяу проксимен әлдеқайда ұзақ болуы мүмкін: бір сыналған орындауда шамамен 60 KB/s жылдамдықпен 22 минут алған.

| Кезең | Не болады? | Doris ішінде сақтала ма? |
|---|---|---|
| Қашықтағы Parquet файлдары | S3 бастапқы жолдары бар 40 файлды сақтайды. | Жоқ |
| S3 TVF нәтижесі | Doris файлдарды уақытша реляциялық нәтиже ретінде оқиды. | Жоқ |
| `events` ішкі кестесі | `INSERT INTO` BE сақтау орнына 10,158,080 жол жазады. | Иә |

Импорттан кейін тексеру сұрауы жол саны мен уақыт аралығын тексереді. Осы мәндер бірге барлық күтілетін жолдар мен оқиға уақытының толық аралығы бар екенін растайды.

In [ ]:
lab.execute("TRUNCATE TABLE events")

lab.insert("""
INSERT INTO events (
    event_time,
    event_id,
    user_id,
    event_type,
    region,
    product_id,
    revenue
)
SELECT
    event_time,
    event_id,
    user_id,
    event_type,
    region,
    product_id,
    revenue
FROM S3(
    "uri" = "s3://doris-public-dataset/doris-course/events/v1/events-*.parquet",
    "s3.endpoint" = "https://s3.us-east-1.amazonaws.com",
    "s3.region" = "us-east-1",
    "s3.credentials_provider_type" = "ANONYMOUS",
    "s3.connection.timeout" = "10000",
    "s3.connection.request.timeout" = "30000",
    "format" = "parquet"
)
""", title="Baseline S3 insert")

lab.verify_baseline("""
SELECT
    COUNT(*) AS row_count,
    MIN(event_time) AS min_event_time,
    MAX(event_time) AS max_event_time
FROM events
""");

<a id="unit-u.d1c6d131eb4647faa49b398961806239"></a>

**Күтілетін нәтиже**

| Тексеру | Күтілетін мән |
|---|---:|
| `row_count` | 10,158,080 |
| `min_event_time` | 2019-12-01 00:00:12 |
| `max_event_time` | 2020-03-11 04:53:08 |

Енгізу нәтижесі **10,158,080 өңделген жол** және кеткен уақыт туралы хабарлайды. Кейін тексеру кестесі тұрақты сақталған нәтижені растайды. Импорт уақыты желі мен Docker ресурстарына байланысты өзгереді және дұрыстық өлшеміне кірмейді.

Енгізу `curlCode: 28, Timeout was reached` қатесімен тоқтаса, Doris 4-бөлімдегі күту уақытында да S3 деректерін оқи алмаған. Компьютеріңізден `s3.us-east-1.amazonaws.com` қолжетімді екенін тексеріңіз (курс README файлындағы Желі тармағын қараңыз), кейін ұяшықты қайталаңыз: `TRUNCATE TABLE` қайталауды қауіпсіз етеді.

<a id="unit-u.a45676fc7dc044f48d663c56ecb91df7"></a>

<a id="7-run-the-first-analytical-query"></a>

## 7. Алғашқы талдамалық сұрауды орындау

Бұл сұрау он миллионнан астам егжей-тегжейлі жолды оқиғалар тізбегінің үш жиынтық нәтижесіне жинақтайды. Ол `GROUP BY`, `COUNT` және `SUM` бар толық талдамалық сұрауды көрсетеді: сатып алу жолдары түсім қосады, ал `view` және `cart` жолдары `0.00` қосады.

In [ ]:
lab.verify_event_counts("""
SELECT
    event_type,
    COUNT(*) AS event_count,
    SUM(revenue) AS total_revenue
FROM events
GROUP BY event_type
ORDER BY event_count DESC
""");


<a id="unit-u.b445a1b2fc124a2eb622c840a01cf365"></a>

**Күтілетін нәтиже**

| Оқиға түрі | Оқиға саны | Жалпы түсім |
|---|---:|---:|
| `view` | 9,437,184 | 0.00 |
| `cart` | 589,824 | 0.00 |
| `purchase` | 131,072 | 39,984,455.64 |

<a id="unit-u.585636e4fbda40b68419fba966bb379f"></a>

<a id="8-practice-find-the-day-with-the-most-purchase-revenue"></a>

## 8. Жаттығу: сатып алу түсімі ең көп күнді табу

Жиындағы 26 күннің тек жетеуінде сатып алулар бар (1-модульде тізімі берілген). Ештеңені орындамас бұрын, осы жеті күннің қайсысында сатып алу түсімі ең көп болғанын болжаңыз: 2019 жылғы желтоқсанда ма, әлде 2020 жылғы наурызда ма?

Содан кейін бос ұяшыққа бір сұрау жазыңыз: тек `purchase` оқиғаларын қалдырып, оларды `DATE(event_time)` арқылы күнтізбелік күн бойынша топтаңыз және түсімі көп күндер алдымен келетіндей реттеңіз. Әр күннің сатып алу саны мен түсімін көрсетіңіз.

In [ ]:
# Your code here

<a id="unit-u.189ed0ced3ad45fa95878aa9e72fde82"></a>

Келесі ұяшықта үлгі шешім бар; ол жиналған күйде. Салыстыру үшін ашыңыз немесе орындаңыз.

In [ ]:
from doris_course.ui import card

revenue_by_day = lab.sql("""
SELECT
    DATE(event_time) AS event_date,
    COUNT(*) AS purchases,
    SUM(revenue) AS revenue
FROM events
WHERE event_type = 'purchase'
GROUP BY DATE(event_time)
ORDER BY revenue DESC
""", title="Purchase revenue by day")

assert len(revenue_by_day) == 7
assert str(revenue_by_day.iloc[0]["event_date"]) == "2020-03-02"
card("Seven purchase days; 2020-03-02 brought the most revenue.", "ok", "Practice checked");

<a id="unit-u.f3d9f1121a95455d96c9c2eed2eaad01"></a>

**Күтілетін нәтиже**

Сатып алу бар әр күнге бір жолдан, барлығы жеті жол және жасыл тексеру белгісі шығады. Бірінші күн **2020-03-02**: **30,546** сатып алу және **9,933,097.98** түсім; соңғы күн 2019-12-04: 9,222 сатып алу және 2,756,102.03 түсім. Наурыздың үш күні алғашқы үш орынды алады. Жеті күннің түсімдері 39,984,455.64 мәніне, яғни 7-бөлімдегі `purchase` жиынтығына қосылады: күн бойынша топтау сол жолдарды басқаша бөледі, бірақ жалпы соманы өзгертпейді.

<a id="unit-u.59b6ede8916a4ff397b3e9221ee5112e"></a>

<a id="stop-the-doris-sandbox"></a>

### Doris сынақ ортасын тоқтату

Контейнердің CPU мен жадын босатқыңыз келсе, осы қосымша ұяшықты орындаңыз. Ол Doris үдерістерін тоқтатады, бірақ контейнерді, бейнені, FE метадеректер томын және BE сақтау томын сақтайды. Бірден 2-зертханалық жұмысқа өтсеңіз, оны өткізіп жіберіңіз.

In [ ]:
lab.shell(r"""
set -euo pipefail

docker stop doris-rta
docker inspect --format 'container={{.State.Status}}' doris-rta
""", title="Stop the Doris sandbox");


<a id="unit-u.50538f78314c4a03a8d412a2dbf6d9a9"></a>

**Күтілетін нәтиже:** Docker `container=exited` көрсетеді. `doris-rta-fe-meta` және `doris-rta-be-storage` атаулы томдары қолжетімді күйде қалады.

<a id="restart-the-doris-sandbox"></a>

### Doris сынақ ортасын қайта іске қосу

Сол сынақ ортасымен жалғастырғыңыз келсе, осы ұяшықты орындаңыз. Ол бар контейнерді іске қосады, Doris сау күйге келгенше күтеді, FE компонентіне қайта қосылады және `events` кестесі тоқтатудан кейін сақталғанын тексереді. Қалпына келтіруді шынымен сынағыңыз келмесе, мұның дәл алдында тоқтату ұяшығын орындамаңыз.

Бұл қайта іске қосу ұяшығы идемпотентті: оны Docker Desktop пен контейнер тоқтаған, іске қосылып жатқан немесе жұмыс істеп тұрған кезде орындауға болады. macOS жүйесінде қажет болса Docker Desktop ашады; Linux жүйесінде ұяшықты орындамас бұрын Docker Engine іске қосыңыз.

In [ ]:
lab.start_container()

lab.connect()
lab.execute("USE doris_course")
lab.verify_recovery("SELECT COUNT(*) AS recovered_rows FROM events");


<a id="unit-u.135154e078734f1badc192d4bc661954"></a>

**Күтілетін нәтиже:** контейнер `healthy` күйіне оралады, ал `recovered_rows` мәні **10,158,080** болады. Бұл FE және BE атаулы томдары зертханалық деректер жиынын сақтағанын растайды.

<a id="unit-u.febbc8387f9e446ca7275c592ed2ea18"></a>

<a id="lab-complete"></a>

## Зертханалық жұмыс аяқталды

Doris жүйесіне MySQL-мен үйлесімді протокол арқылы қосылдыңыз, FE және BE тексердіңіз, S3 TVF арқылы қашықтағы Parquet файлдарына сұрау жасадыңыз, ішкі кестеге 10,158,080 жол жүктедіңіз, талдамалық агрегаттау орындадыңыз және деректер жиынын сақтау үшін Docker атаулы томдарын баптадыңыз.

Қосымша жұмыс: [Metabase қосып, бақылау тақтасын құру](optional01_metabase_dashboard.ipynb)

Ресми анықтамалар: [All-In-One бейнесі](https://doris.apache.org/community/developer-guide/all-in-one-image/) · [S3 TVF](https://doris.apache.org/docs/4.x/sql-manual/sql-functions/table-valued-functions/s3/) · [CREATE TABLE](https://doris.apache.org/docs/4.x/sql-manual/sql-statements/table-and-view/table/CREATE-TABLE/)

<a id="unit-u.7a936acc59db482e9e5623e278daacee"></a>

<a id="next"></a>

## Келесі қадам

[1-тест: Apache Doris жүйесіне кіріспе](quiz01_doris_fundamentals.ipynb) ноутбугін ашып, кейін [2-модуль: Apache Doris архитектурасын тереңірек зерттеу](../module02-architecture/course.md) бөліміне өтіңіз.